# Video to Inverse Dynamics Pipeline

This notebook shows the full staged path from single-camera video to inverse dynamics. It includes OpenSim preflight checks, estimated external loads, and places to inspect outputs before trusting downstream kinetics.

## 1. Import and set paths

Edit these values first. OpenSim stages require OpenSim-compatible Python bindings.

In [ ]:
from pathlib import Path
import monomech as mm

VIDEO_PATH = Path("data/subject01.mp4")
OUTPUT_DIR = Path("outputs/subject01_full_pipeline")
BODY_MASS_KG = 75.0

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
MODEL_PATH = mm.get_builtin_osim_model("pose")

print("Model:", MODEL_PATH)
print("Available models:", mm.list_builtin_osim_models())

## 2. Video to pose and TRC

The pipeline exports CSV files and a TRC file. Inspect the pose summary before continuing.

In [ ]:
trial = mm.load_video(VIDEO_PATH)

run = trial.run_pipeline(
    export_csv=True,
    export_trc=True,
    output_dir=OUTPUT_DIR / "pose",
)

display(run.pose2d.summary().head(12))
display(run.pose3d_global.to_wide_df().head())
print("TRC:", run.trc_path)

## 3. Scale the model

By default, monomech checks the TRC for NaNs before OpenSim runs. If gaps are found, it writes an `*_opensim_ready.trc` file and records the fill report in metadata.

In [ ]:
scale = trial.run_opensim_scale(
    model_path=MODEL_PATH,
    trc_path=run.trc_path,
    output_dir=OUTPUT_DIR / "scale",
)

display(scale.summary())
print("Scale preflight:", scale.metadata.get("preflight"))

## 4. Run inverse kinematics

In [ ]:
ik = trial.run_opensim_ik(
    model_path=scale.scaled_model_path,
    trc_path=run.trc_path,
    output_dir=OUTPUT_DIR / "ik",
)

print("IK path:", ik.path)
display(ik.to_dataframe().head())
print("IK preflight:", ik.metadata.get("preflight"))

## 5. Estimate external loads

Measured force plates are preferred for kinetics. This estimated GRF helper is useful for examples and exploratory work when measured forces are not available.

In [ ]:
estimated_loads = mm.external.estimate_grf(
    pose3d=run.pose3d_global,
    body_mass_kg=BODY_MASS_KG,
    sides=("left", "right"),
)

for load in estimated_loads:
    print(load.name, load.applied_to_body, load.source, load.is_estimated)
    display(load.to_dataframe().head())

## 6. Run inverse dynamics

The ID helper aligns external loads to the IK time vector and sanitizes coordinate NaNs by default. Inspect `coordinate_preflight` and generated external-load paths afterward.

In [ ]:
id_result = trial.run_opensim_id(
    model_path=scale.scaled_model_path,
    ik_path=ik.path,
    external_forces=estimated_loads,
    output_dir=OUTPUT_DIR / "id",
)

print("ID path:", id_result.path)
print("Coordinate preflight:", id_result.metadata.get("coordinate_preflight"))
print("External loads XML:", id_result.metadata.get("external_loads_xml_path"))
print("External loads MOT:", id_result.metadata.get("external_loads_mot_path"))
display(id_result.to_dataframe().head())

## 7. Strict mode when debugging

Automatic sanitizing helps OpenSim run, but strict mode is better when you want to catch every data issue.

In [ ]:
from monomech import OpenSimIKConfig, OpenSimIDConfig

# ik_strict = trial.run_opensim_ik(
#     model_path=scale.scaled_model_path,
#     trc_path=run.trc_path,
#     config=OpenSimIKConfig(sanitize_marker_data=False),
# )
#
# id_strict = trial.run_opensim_id(
#     model_path=scale.scaled_model_path,
#     ik_path=ik.path,
#     external_forces=estimated_loads,
#     config=OpenSimIDConfig(sanitize_coordinates=False),
# )